<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/sql_cleaning_500rows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas openpyxl

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving SQLite_Cleaning_Practice_500_Rows.xlsx to SQLite_Cleaning_Practice_500_Rows.xlsx


In [ ]:
import pandas as pd
df=pd.read_excel("SQLite_Cleaning_Practice_500_Rows.xlsx")
print(df.head())

   Customer_ID           Name  Gender   Age                      Email  \
0            1      Anita Raj    Male  38.0      anita.raj@example.com   
1            2      Mohan Raj    Male  58.0      mohan.raj@example.com   
2            3  Karthik Kumar    Male  56.0  karthik.kumar@example.com   
3            4    Nisha Kumar  Female  21.0    nisha.kumar@example.com   
4            5     Meena Devi    Male  42.0     meena.devi@example.com   

        Phone        City Department Salary  Join_Date  
0  9362950628     Madurai         HR  38434 2024-10-17  
1  9553035110  Coimbatore         IT  37280 2022-03-24  
2  9313500298        Pune      Sales  53893 2023-07-09  
3  9914763202   Bangalore      Sales  69597 2022-07-24  
4  9209747451     Chennai      Sales  37676 2023-01-06  


In [ ]:
import sqlite3
conn=sqlite3.connect("customer_cleaning.db")
cursor=conn.cursor()

In [ ]:
df.to_sql("customer_raw",conn,if_exists="replace",index=False)

500

In [ ]:
print(pd.read_sql_query("select count(*) as total_rows from customer_raw",conn))

   total_rows
0         500


In [ ]:
print(pd.read_sql_query("""
SELECT
    SUM(CASE WHEN name IS NULL OR TRIM(name) = '' THEN 1 ELSE 0 END) AS missing_names,
    SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age,
    SUM(CASE WHEN email IS NULL OR TRIM(email) = '' THEN 1 ELSE 0 END) AS missing_email,
    SUM(CASE WHEN join_date IS NULL THEN 1 ELSE 0 END) AS missing_join_date
FROM customer_raw
""", conn))

   missing_names  missing_age  missing_email  missing_join_date
0              0            5              0                  5


In [ ]:
conn.execute("DROP TABLE IF EXISTS customer_clean")

conn.execute("""
CREATE TABLE customer_clean AS
SELECT
    Customer_id,
    TRIM(name) AS Name,

    CASE
        WHEN LOWER(TRIM(gender)) = 'male' THEN 'male'
        WHEN LOWER(TRIM(gender)) = 'female' THEN 'female'
        ELSE NULL
    END AS Gender,

    CASE
        WHEN Age BETWEEN 18 AND 100 THEN Age
        ELSE NULL
    END AS Age,

    LOWER(TRIM(Email)) AS Email,
    TRIM(Phone) AS Phone,
    UPPER(TRIM(Department)) AS Department,

    CAST(REPLACE(Salary, ',', '') AS INTEGER) AS Salary,

    Join_Date

FROM customer_raw
""")
df=pd.read_sql("select * from customer_clean",conn)

In [ ]:

print(pd.read_sql_query("""
select Customer_id,Email from customer_clean  where Email not like'%@%.%'
or Email like '%%'
""",conn))

     Customer_ID                      Email
0              1      anita.raj@example.com
1              2      mohan.raj@example.com
2              3  karthik.kumar@example.com
3              4    nisha.kumar@example.com
4              5     meena.devi@example.com
..           ...                        ...
495          496      anita.raj@example.com
496          497     priya.devi@example.com
497          498   suresh.kumar@example.com
498          499  saravanan.raj@example.com
499          500     priya.devi@example.com

[500 rows x 2 columns]


In [ ]:
conn.execute(""" update customer_clean set Email=null where Email not like '%@%.%' or Email like '%%'""" )

In [ ]:
cursor.execute("select* from customer_clean")
print(df.head())

   Customer_ID           Name  Gender   Age                      Email  \
0            1      Anita Raj    male  38.0      anita.raj@example.com   
1            2      Mohan Raj    male  58.0      mohan.raj@example.com   
2            3  Karthik Kumar    male  56.0  karthik.kumar@example.com   
3            4    Nisha Kumar  female  21.0    nisha.kumar@example.com   
4            5     Meena Devi    male  42.0     meena.devi@example.com   

        Phone Department  Salary            Join_Date  
0  9362950628         HR   38434  2024-10-17 00:00:00  
1  9553035110         IT   37280  2022-03-24 00:00:00  
2  9313500298      SALES   53893  2023-07-09 00:00:00  
3  9914763202      SALES   69597  2022-07-24 00:00:00  
4  9209747451      SALES   37676  2023-01-06 00:00:00  


In [ ]:
print(pd.read_sql_query(""" select Customer_id ,Phone from customer_clean where length(phone)<>10""",conn))

Empty DataFrame
Columns: [Customer_ID, Phone]
Index: []


In [ ]:
print(pd.read_sql_query("""
select Name,Email,Phone,count(*) as duplicated_count from Customer_clean group by Name,Email,Phone having count(*) > 1""",conn))

          Name                    Email       Phone  duplicated_count
0    Anita Raj    anita.raj@example.com  9245668163                 2
1  Deepa Kumar  deepa.kumar@example.com  9793773681                 2
2    Divya Raj    divya.raj@example.com  9754049436                 2
3  Nisha Kumar  nisha.kumar@example.com  9366493008                 2
4   Priya Devi   priya.devi@example.com  9471178705                 2


In [ ]:
print(pd.read_sql_query("""
select Customer_id,count(*) as count_id from customer_clean group by Customer_id having count(*)>1""",conn))

Empty DataFrame
Columns: [Customer_ID, count_id]
Index: []


In [ ]:
print(pd.read_sql_query(
    "select count(*) as final_rows from customer_clean",conn
))
print(pd.read_sql_query("select * from customer_clean limit 10",conn))
conn.commit()
conn.close

   final_rows
0         500
   Customer_ID           Name  Gender   Age                      Email  \
0            1      Anita Raj    male  38.0      anita.raj@example.com   
1            2      Mohan Raj    male  58.0      mohan.raj@example.com   
2            3  Karthik Kumar    male  56.0  karthik.kumar@example.com   
3            4    Nisha Kumar  female  21.0    nisha.kumar@example.com   
4            5     Meena Devi    male  42.0     meena.devi@example.com   
5            6      Rahul Raj  female  23.0      rahul.raj@example.com   
6            7     Sneha Devi    male  56.0     sneha.devi@example.com   
7            8     Ravi Kumar    male  35.0     ravi.kumar@example.com   
8            9      Anita Raj  female  38.0      anita.raj@example.com   
9           10      Divya Raj  female  25.0      divya.raj@example.com   

        Phone  Department  Salary            Join_Date  
0  9362950628          HR   38434  2024-10-17 00:00:00  
1  9553035110          IT   37280  2022-03-

<function Connection.close()>